In [1]:
# ==============================================================================
# 1. USER SELECTION PARAMETERS (Tag this cell as "Parameter cell")
# ==============================================================================
SELECTED_MAIN_CATEGORY = "Devotional"
# Pipelines pass parameters as text, so default to a string representation of an empty list
SELECTED_SUB_CATEGORIES = "[]" 

StatementMeta(, 61bf4b84-b73b-4749-8041-ec554c5e2460, 3, Finished, Available, Finished, False)

In [ ]:
# %pip install pymupdf
# %pip install reportlab

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
import pymupdf as fitz
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
import io
import re
import os
import ast

# ==============================================================================
# 1.5 PIPELINE PARAMETER PARSING & FILENAME GENERATION
# ==============================================================================
# Safely convert the string representation of the list back into an actual Python list
if isinstance(SELECTED_SUB_CATEGORIES, str):
    SELECTED_SUB_CATEGORIES = ast.literal_eval(SELECTED_SUB_CATEGORIES)

# Dynamically generate base filename based on selection
if SELECTED_SUB_CATEGORIES:
    # Remove spaces and increase length to 60 for long parameter lists
    clean_list = [cat.replace(" ", "") for cat in SELECTED_SUB_CATEGORIES]
    sub_cat_str = "_".join(clean_list)[:60] 
    base_pdf_path = f"/lakehouse/default/Files/Generated_PDFs/{SELECTED_MAIN_CATEGORY}_{sub_cat_str}_Book.pdf"
else:
    base_pdf_path = f"/lakehouse/default/Files/Generated_PDFs/{SELECTED_MAIN_CATEGORY}_Complete_Book.pdf"

# --- NEW LOGIC: Unique Filename Generation ---
def get_unique_filepath(filepath):
    """Checks if a file exists and appends a numeric suffix (_1, _2) if needed."""
    if not os.path.exists(filepath):
        return filepath
    
    base, ext = os.path.splitext(filepath)
    counter = 1
    new_filepath = f"{base}_{counter}{ext}"
    
    while os.path.exists(new_filepath):
        counter += 1
        new_filepath = f"{base}_{counter}{ext}"
        
    return new_filepath

# Apply the function to get the final safe output path
OUTPUT_PDF_PATH = get_unique_filepath(base_pdf_path)

# ==============================================================================
# 2. QUERY GOLD LAYER & TRANSLATE PATHS
# ==============================================================================
def get_songs_from_gold_layer(main_cat, sub_cats_list):
    """Queries the Star Schema dynamically and maps the CSV path to the source PDF."""
    
    query = f"""
        SELECT 
            f.Title as title, 
            c.Sub_Category as category, 
            COALESCE(s.Scale, '-') as scale, 
            COALESCE(f.Singer, f.Composer, 'Unknown') as details, 
            f.Page_No as original_page, 
            f.Source_Path
        FROM Fact_Songs f
        LEFT JOIN Dim_Category c ON f.Category_ID = c.Category_ID
        LEFT JOIN Dim_Scale s ON f.Scale_ID = s.Scale_ID
        WHERE c.Main_Category = '{main_cat}' 
    """
    
    if sub_cats_list:
        sub_cats_sql = ", ".join([f"'{c}'" for c in sub_cats_list])
        query += f" AND c.Sub_Category IN ({sub_cats_sql})"
        
    query += " ORDER BY c.Sub_Category, f.Title"
    
    df = spark.sql(query).collect()
    
    songs = []
    for idx, row in enumerate(df):
        # 1. Translate cloud path to local path
        local_path = re.sub(r"abfss://[^/]+/[^/]+/Files/", "/lakehouse/default/Files/", row.Source_Path)
        
        # 2. Strip the "?version=" and "?flength=" system garbage
        clean_path = local_path.split("?")[0]
        
        # 3. Swap from .csv to the matching .pdf file
        folder_path = os.path.dirname(clean_path)
        csv_filename = os.path.basename(clean_path)
        
        # Extracts the core prefix (e.g., 'Krishna' from 'Krishna_index.csv')
        core_name = csv_filename.split('_')[0].split('.')[0] 
        
        pdf_path = clean_path # Fallback
        
        # Look in the folder and find the PDF that starts with the core name
        if os.path.exists(folder_path):
            for file in os.listdir(folder_path):
                if file.startswith(core_name) and file.lower().endswith(".pdf"):
                    pdf_path = os.path.join(folder_path, file)
                    break
        
        songs.append({
            "title": row.title,
            "category": row.category,
            "scale": row.scale,
            "details": row.details,
            "original_page": int(row.original_page),
            "source_path": pdf_path,
            "new_page_index": idx 
        })
        
    return songs

# ==============================================================================
# 3. REPORTLAB INDEX GENERATION
# ==============================================================================
def build_index_canvas(songs, index_page_offset=0):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=A4)
    width, height = A4
    link_rects = []
    
    def draw_headers():
        c.setFont("Helvetica-Bold", 16)
        c.drawCentredString(width / 2.0, height - 40, f"{SELECTED_MAIN_CATEGORY} Collection")
        c.setFont("Helvetica-Bold", 11)
        y = height - 80
        
        c.drawString(50, y, "Category / Song Title")
        c.drawString(260, y, "Scale")
        c.drawString(340, y, "Singer / Composer")
        c.drawString(500, y, "Page")
        c.line(50, y - 5, 540, y - 5)
        return y - 25

    y = draw_headers()
    current_idx_page = 0
    current_category = None

    for song in songs:
        if y < 60:
            c.showPage()
            current_idx_page += 1
            y = draw_headers()
            current_category = None 
            
        if song["category"] != current_category:
            y -= 10
            if y < 60:
                c.showPage()
                current_idx_page += 1
                y = draw_headers()
            
            c.setFont("Helvetica-Bold", 11)
            c.drawString(50, y, f"--- {song['category']} ---")
            y -= 22
            current_category = song["category"]
            
        c.setFont("Helvetica", 11)
        target_page_absolute = song["new_page_index"] + index_page_offset + 1
        
        display_details = song["details"]
        if len(display_details) > 22:
            display_details = display_details[:19] + "..."
        
        c.drawString(60, y, song["title"])
        c.drawString(260, y, song["scale"])
        c.drawString(340, y, display_details)
        c.drawString(500, y, str(target_page_absolute))
        
        pymupdf_y0 = height - y - 12
        pymupdf_y1 = height - y + 5
        rect = fitz.Rect(50, pymupdf_y0, 540, pymupdf_y1)
        
        link_rects.append((current_idx_page, rect, song["new_page_index"]))
        y -= 22
        
    c.save()
    packet.seek(0)
    return packet, link_rects, current_idx_page + 1

# ==============================================================================
# 4. PYMUPDF STITCHING & HYPERLINKING
# ==============================================================================
def build_indexed_pdf():
    songs_metadata = get_songs_from_gold_layer(SELECTED_MAIN_CATEGORY, SELECTED_SUB_CATEGORIES)
    
    if not songs_metadata:
        print(f"No songs found for selection. Aborting.")
        return

    output_doc = fitz.open()

    _, _, total_index_pages = build_index_canvas(songs_metadata, index_page_offset=0)
    index_pdf_bytes, link_rects, _ = build_index_canvas(songs_metadata, index_page_offset=total_index_pages)
    index_doc = fitz.open("pdf", index_pdf_bytes.read())

    output_doc.insert_pdf(index_doc)
    
    for song in songs_metadata:
        try:
            src_doc = fitz.open(song["source_path"])
            page_to_extract = song["original_page"] - 1 
            output_doc.insert_pdf(src_doc, from_page=page_to_extract, to_page=page_to_extract)
            src_doc.close()
        except Exception as e:
            print(f"Error extracting page {song['original_page']} from {song['source_path']}: {e}")

    for index_page_num, rect, target_song_index in link_rects:
        page_obj = output_doc[index_page_num]
        link_data = {
            "kind": fitz.LINK_GOTO,
            "page": target_song_index + total_index_pages, 
            "from": rect
        }
        page_obj.insert_link(link_data)

    for page_num in range(total_index_pages, len(output_doc)):
        page_obj = output_doc[page_num]
        btn_rect = fitz.Rect(400, 15, 560, 40)
        
        page_obj.insert_textbox(
            btn_rect, 
            "Go to Index", 
            fontsize=12, 
            fontname="helv", 
            color=(0, 0, 1), 
            align=fitz.TEXT_ALIGN_RIGHT
        )
        page_obj.insert_link({"kind": fitz.LINK_GOTO, "page": 0, "from": btn_rect})

    os.makedirs(os.path.dirname(OUTPUT_PDF_PATH), exist_ok=True)
    output_doc.save(OUTPUT_PDF_PATH)
    print(f"✅ Generated Complete Custom PDF Book at: {OUTPUT_PDF_PATH}")

    output_doc.close()
    index_doc.close()

# Execute
build_indexed_pdf()

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# Instantly release Spark compute resources to prevent pipeline capacity errors
spark.stop()

StatementMeta(, , -1, Cancelled, , Cancelled, True)